# 🏥 GIEndoDL: TASK D47 - HUẤN LUYỆN SẢN XUẤT (PRODUCTION RUN 5 - 50 EPOCHS)
## Đề tài: Ứng dụng Học Sâu trong Nhận diện và Phân loại Tổn thương Đường Tiêu hóa từ Ảnh Nội soi
**Học viên thực hiện:** Lê Huy Phát | **Đơn vị:** Đại học Công Thương TP.HCM (HUIT)  
**Mô hình Đề xuất:** Hybrid CNN-CBAM-Transformer (ResNet-50 + CBAM Stage 4 + 2-layer Transformer + GAP Head)  
**Bộ Dữ liệu:** Schema 17 phân lớp y sinh sạch (Zero Video Leakage, Smart Aspect-Ratio Letterbox, Class-Balanced Focal Loss)  
**Mục tiêu Task D47:** Huấn luyện hoàn chỉnh 50 Epochs trên Kaggle GPU (Tesla T4/P100), xuất trọn bộ 24 tệp kết quả chuẩn mực y khoa (Best/Last Models, Clinical Report, 17x17 Confusion Matrices, Error Cases, Grad-CAM, Convergence Curves, ZIP 1-click download).

---
### 📋 Cấu hình Kỹ thuật SOTA Được Kế Thừa:
1. **Không gian nhãn:** 17 Phân lớp chuẩn y sinh quốc tế (Mayo Colitis 3 mức, Barrett gộp, loại bỏ lớp rác $< 15$ mẫu).
2. **Phân hoạch:** GroupStratifiedKFold (7,452 Train / 1,601 Val / 1,591 Test) với rò rỉ bệnh nhân $\Lambda = 0.00000000\%$.
3. **Tiền xử lý:** Cắt viền Otsu, che text overlay, giảm phản quang mềm, LAB CLAHE, bù viền Letterbox 0% méo.
4. **Tăng cường y sinh:** Policy D (Flip, Rotate, Lens Dome Distortion, Peristaltic Blur, Narrow ColorJitter, No CutMix/MixUp).
5. **Hàm mất mát:** Class-Balanced Focal Loss ($\beta=0.75, \gamma=2.0, \epsilon=0.05$).
6. **Bộ lấy mẫu:** WeightedRandomSampler với tham số lũy thừa $\beta=0.75$.
7. **Lịch trình LR:** CosineAnnealingLR (T_max=50, eta_min=1e-6) kết hợp tối ưu hóa AdamW ($10^{-4}$).
8. **Đầu ra lâm sàng:** Trọn bộ 24 file đối sánh đầy đủ tương đương Run 4.

In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU & MÔI TRƯỜNG KAGGLE
!nvidia-smi

import os
import sys
import time
from pathlib import Path
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 80)
print(f'🚀 PyTorch Version: {torch.__version__}')
print(f'⚡ Device: {device} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎮 GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 80)

In [ ]:
# 2. CÀI ĐẶT CÁC THƯ VIỆN BỔ TRỢ
!pip install -q albumentations timm scikit-learn matplotlib seaborn tqdm

In [ ]:
# 3. CẬP NHẬT MÃ NGUỒN MỚI NHẤT TỪ GITHUB
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/GIEndoDL")
if not REPO_DIR.exists():
    print("⚡ Đang clone mã nguồn GIEndoDL từ GitHub...")
    subprocess.run(["git", "clone", "https://github.com/LeHuyPhat2907/GIEndoDL.git", str(REPO_DIR)], check=True)

%cd /kaggle/working/GIEndoDL
print("⚡ Đang kéo các bản cập nhật mới nhất từ GitHub...")
!git pull origin main
print(f"📂 Thư mục làm việc hiện tại: {Path.cwd()}")

In [ ]:
# 4. KIỂM TRA BỘ DỮ LIỆU HYPERKVASIR (NẾU CHƯA CÓ, TỰ ĐỘNG TẢI TỪ SIMULA)
raw_dir = Path("/kaggle/working/GIEndoDL/data/raw/labeled-images")
if not raw_dir.exists() or len(list(raw_dir.glob('*/*/*.jpg'))) < 1000:
    # Kiểm tra trong /kaggle/input
    has_input = False
    if Path("/kaggle/input").exists():
        for root, _, files in os.walk("/kaggle/input"):
            if any(f.lower().endswith('.jpg') for f in files):
                has_input = True
                print(f"✅ Đã tìm thấy dữ liệu ảnh trong /kaggle/input!")
                break
    if not has_input:
        print("🌐 Dữ liệu chưa tồn tại, bắt đầu tải trực tiếp từ máy chủ Simula (3.9 GB)...")
        !python src/dataset/download_hyperkvasir.py
else:
    print(f"✅ Dữ liệu HyperKvasir đã có sẵn sàng tại: {raw_dir}")

In [ ]:
# 5. THỰC THI HUẤN LUYỆN SẢN XUẤT 50 EPOCHS (PRODUCTION RUN 5)
!python src/training/train_production_run5.py \
    --device cuda \
    --epochs 50 \
    --batch_size 32 \
    --img_size 224                 --fast \
    --lr 1e-4 \
    --weight_decay 1e-2 \
    --beta 0.75 \
    --gamma 2.0 \
    --label_smoothing 0.05 \
    --num_workers 4 \
    --output_dir outputs/optimal_hybrid_model_run5

In [ ]:
# 6. HIỂN THỊ TRỰC QUAN HÓA KẾT QUẢ ĐỐI SÁNH TỐI ƯU & MA TRẬN NHẦM LẪN
from IPython.display import Image, display, Markdown

out_dir = Path("/kaggle/working/GIEndoDL/outputs/optimal_hybrid_model_run5")

print("📊 ĐỒ THỊ ĐỐI SÁNH TỐI ƯU 4 GÓC ĐỘ (LOSS, MACRO F1, ACCURACY, FPS):")
if (out_dir / "optimal_training_convergence_curves.png").exists():
    display(Image(filename=str(out_dir / "optimal_training_convergence_curves.png")))

print("📊 MA TRẬN NHẦM LẪN CHUẨN HÓA (17x17 CONFUSION MATRIX):")
if (out_dir / "confusion_matrix_normalized.png").exists():
    display(Image(filename=str(out_dir / "confusion_matrix_normalized.png")))

print("📊 BẢNG XẾP HẠNG F1-SCORE TRÊN 17 PHÂN LỚP BỆNH HỌC:")
if (out_dir / "per_class_f1_ranking.png").exists():
    display(Image(filename=str(out_dir / "per_class_f1_ranking.png")))

print("📊 BIỂU ĐỒ ĐỐI CHIẾU PRECISION VS RECALL:")
if (out_dir / "per_class_precision_recall.png").exists():
    display(Image(filename=str(out_dir / "per_class_precision_recall.png")))

In [ ]:
# 7. HIỂN THỊ BÁO CÁO LÂM SÀNG TOÀN DIỆN (CLINICAL REPORT)
report_file = out_dir / "clinical_report.md"
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("Chưa tìm thấy clinical_report.md")

In [ ]:
# 8. TẠO LIÊN KẾT TẢI VỀ 1-CLICK CHO FILE ZIP TOÀN BỘ KẾT QUẢ
from IPython.display import FileLink, display
import os
from pathlib import Path

zip_file = Path("/kaggle/working/GIEndoDL/outputs/optimal_hybrid_model_run5_results.zip")
if not zip_file.exists():
    zip_file = Path("/kaggle/working/optimal_hybrid_model_run5_results.zip")

print('=' * 85)
if zip_file.exists():
    print(f"📦 TẬP TIN NÉN KẾT QUẢ ĐÃ SẴN SÀNG: {zip_file.name} ({zip_file.stat().st_size / (1024**2):.2f} MB)")
    print("📥 Bấm vào đường dẫn bên dưới để tải trực tiếp toàn bộ 24 file về máy:")
    if Path("outputs/optimal_hybrid_model_run5_results.zip").exists():
        display(FileLink("outputs/optimal_hybrid_model_run5_results.zip"))
    elif Path("optimal_hybrid_model_run5_results.zip").exists():
        display(FileLink("optimal_hybrid_model_run5_results.zip"))
    else:
        display(FileLink(str(zip_file)))
else:
    print("Không tìm thấy file zip kết quả!")
print('=' * 85)